# Group-specific errors in partisan exposure

This notebook computes the group-specific Mean Signed Deviation (MSD) reported in
main-text Fig. 5, SI Fig. S5, and SI Table S11. The analysis uses nomination
networks and retains groups represented by at least five personas.


In [ ]:
import os
from pathlib import Path

BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "code":
    BASE_DIR = BASE_DIR.parent

DATA_DIR = BASE_DIR / "outputs" / "data"
FIGURE_DIR = BASE_DIR / "outputs" / "figures"
OUT_DIR = FIGURE_DIR / "demographic_exposure_errors"
IMAGE_DIR = BASE_DIR / "code" / "images"
TABLE_DIR = BASE_DIR / "code" / "tables"
FONT_DIR = BASE_DIR / "font"
MPL_CONFIG_DIR = BASE_DIR / "outputs" / ".mplconfig"
for folder in (DATA_DIR, OUT_DIR, IMAGE_DIR, TABLE_DIR, MPL_CONFIG_DIR):
    folder.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CONFIG_DIR))

import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
barlow_files = sorted(FONT_DIR.glob("Barlow-*.ttf"))
if not barlow_files:
    raise FileNotFoundError(f"Barlow font files not found in {FONT_DIR}")
for font_path in barlow_files:
    font_manager.fontManager.addfont(str(font_path))
plt.rcParams.update({
    "font.family": "Barlow",
    "font.weight": "normal",
    "axes.labelweight": "normal",
    "axes.titleweight": "normal",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.dpi": 110,
})
pd.set_option("display.max_columns", 120)


In [ ]:
ANES_CSV = BASE_DIR / "data" / "anes_100.csv"
EXPOSURE_LONG_CSV = DATA_DIR / "local_evaluation_exposure_long_sample_100_v3_by_network_view.csv"
ONE_WAY_SUMMARY_CSV = DATA_DIR / "demographic_exposure_error_by_category_sample_100_v3_by_network_view.csv"
SIGNED_RANKING_CSV = DATA_DIR / "largest_signed_demographic_errors_sample_100_v3_by_network_view.csv"

PRIMARY_NETWORK_VIEW = "directed_out"
NETWORK_VIEWS = [PRIMARY_NETWORK_VIEW]
METHOD_ORDER = ["c1", "c5", "c10", "c25", "c50"]
MIN_PERSONAS = 5
PARTY_LABELS = {"dem": "Democratic exposure", "rep": "Republican exposure"}
VIEW_LABELS = {"directed_out": "Directed-out"}

DEMOGRAPHIC_ATTRIBUTES = {
    "profile_gender": "Gender",
    "profile_age_bucket": "Age group",
    "profile_racethnicity": "Race/ethnicity",
    "profile_educ5": "Education",
    "profile_region4": "Region",
    "profile_relig": "Religion",
}

CONFIG_COLUMNS = [
    "network_view", "model", "model_label", "temperature", "chunk_size",
    "method", "attr_label", "attribute_order", "party",
]

def standard_error(series):
    values = pd.to_numeric(series, errors="coerce").dropna()
    return values.std(ddof=1) / np.sqrt(len(values)) if len(values) > 1 else np.nan


## Load and validate the persona-level exposure data

Notebook 03 already defines exposure, missingness, party normalization, and the
five-category mapping. Notebook 04 therefore only attaches demographic attributes;
it does not recompute exposure.


In [ ]:
survey = pd.read_csv(ANES_CSV, encoding="mac_roman", low_memory=False)
exposure = pd.read_csv(EXPOSURE_LONG_CSV)
exposure = exposure[exposure["network_view"].eq(PRIMARY_NETWORK_VIEW)].copy()

required_exposure_columns = {
    "network_view", "model", "model_label", "temperature", "chunk_size",
    "method", "attr_label", "attribute_order", "run_number", "run_id",
    "caseid", "ego_party", "party", "llm_pct", "llm_cat", "survey_cat",
    "n_neighbors", "n_valid_party_neighbors",
}
missing_columns = required_exposure_columns.difference(exposure.columns)
assert not missing_columns, f"Missing exposure columns: {sorted(missing_columns)}"
assert set(DEMOGRAPHIC_ATTRIBUTES).issubset(survey.columns)

survey["caseid"] = pd.to_numeric(survey["caseid"], errors="raise").astype(int)
exposure["caseid"] = pd.to_numeric(exposure["caseid"], errors="raise").astype(int)
for column in ["llm_cat", "survey_cat", "llm_pct", "temperature"]:
    exposure[column] = pd.to_numeric(exposure[column], errors="coerce")
exposure["method"] = pd.Categorical(
    exposure["method"].astype(str), METHOD_ORDER, ordered=True
)
exposure["signed_error"] = exposure["llm_cat"] - exposure["survey_cat"]
exposure["abs_error"] = exposure["signed_error"].abs()
exposure["valid_pair"] = exposure[["llm_cat", "survey_cat"]].notna().all(axis=1)
exposure["cat_match"] = np.where(
    exposure["valid_pair"], exposure["llm_cat"].eq(exposure["survey_cat"]), np.nan
)

assert len(exposure) == 200_000
assert set(exposure["network_view"]) == set(NETWORK_VIEWS)
network_counts = (
    exposure.drop_duplicates(["network_view", "model", "run_id"])
    .groupby("network_view").size().to_dict()
)
assert network_counts == {"directed_out": 1000}
assert survey["caseid"].nunique() == 100

demographic_columns = ["caseid", *DEMOGRAPHIC_ATTRIBUTES]
persona_demographics = survey[demographic_columns].copy()
assert persona_demographics["caseid"].is_unique
exposure_demo = exposure.merge(
    persona_demographics, on="caseid", how="left", validate="many_to_one"
)
assert exposure_demo[list(DEMOGRAPHIC_ATTRIBUTES)].notna().all().all()

print(f"Loaded {len(exposure):,} persona-target observations.")
print("Unique generated networks:", network_counts)
display(
    exposure.groupby("network_view")
    .agg(
        valid_pair_rate=("valid_pair", "mean"),
        mean_neighbors=("n_neighbors", "mean"),
    ).round(3)
)


## Reshape demographic attributes and calculate run-level errors


In [ ]:
base_columns = [
    *CONFIG_COLUMNS, "run_number", "run_id", "caseid", "ego_party",
    "llm_pct", "llm_cat", "survey_cat", "signed_error", "abs_error",
    "cat_match", "valid_pair", "n_neighbors", "n_valid_party_neighbors",
]

long_parts = []
for attribute, attribute_label in DEMOGRAPHIC_ATTRIBUTES.items():
    part = exposure_demo[base_columns + [attribute]].copy()
    part = part.rename(columns={attribute: "category"})
    part["attribute"] = attribute
    part["attribute_label"] = attribute_label
    part["category"] = part["category"].astype(str).str.strip()
    part = part[part["category"].ne("") & part["category"].ne("nan")]
    long_parts.append(part)
demographic_long = pd.concat(long_parts, ignore_index=True)


def summarize_at_run_level(frame, group_columns):
    run_columns = group_columns + ["run_id"]
    run_level = (
        frame.groupby(run_columns, dropna=False, observed=True)
        .agg(
            run_n_personas_total=("caseid", "nunique"),
            run_n_valid_personas=("valid_pair", "sum"),
            run_mean_signed_error=("signed_error", "mean"),
            run_mean_abs_error=("abs_error", "mean"),
            run_match_rate=("cat_match", "mean"),
            run_mean_llm_pct=("llm_pct", "mean"),
            run_mean_llm_cat=("llm_cat", "mean"),
            run_mean_survey_cat=("survey_cat", "mean"),
        ).reset_index()
    )
    summary = (
        run_level.groupby(group_columns, dropna=False, observed=True)
        .agg(
            n_runs=("run_id", "nunique"),
            n_runs_with_valid_pairs=("run_mean_abs_error", "count"),
            mean_valid_personas=("run_n_valid_personas", "mean"),
            mean_signed_error=("run_mean_signed_error", "mean"),
            se_signed_error=("run_mean_signed_error", standard_error),
            mean_abs_error=("run_mean_abs_error", "mean"),
            se_abs_error=("run_mean_abs_error", standard_error),
            match_rate=("run_match_rate", "mean"),
            se_match_rate=("run_match_rate", standard_error),
            mean_llm_pct=("run_mean_llm_pct", "mean"),
            mean_llm_cat=("run_mean_llm_cat", "mean"),
            mean_survey_cat=("run_mean_survey_cat", "mean"),
        ).reset_index()
    )
    total_counts = (
        frame.groupby(group_columns, dropna=False, observed=True)["caseid"]
        .nunique().rename("n_personas_total").reset_index()
    )
    valid_counts = (
        frame[frame["valid_pair"]]
        .groupby(group_columns, dropna=False, observed=True)["caseid"]
        .nunique().rename("n_personas_valid").reset_index()
    )
    summary = summary.merge(total_counts, on=group_columns, how="left")
    summary = summary.merge(valid_counts, on=group_columns, how="left")
    summary["n_personas_valid"] = summary["n_personas_valid"].fillna(0).astype(int)
    return run_level, summary


one_way_groups = CONFIG_COLUMNS + ["attribute", "attribute_label", "category"]
one_way_run, one_way_summary = summarize_at_run_level(demographic_long, one_way_groups)

assert one_way_summary["n_runs"].eq(10).all()
print(f"Run-level subgroup records: {len(one_way_run):,}")
print(f"Configuration-level subgroup summaries: {len(one_way_summary):,}")


## Configuration-level group-specific MSD

Retain demographic groups represented by at least five personas and export the
configuration-level summary used to construct SI Table S11.


In [ ]:
eligible_one_way = one_way_summary[
    one_way_summary["mean_valid_personas"].ge(MIN_PERSONAS)
].copy()
one_way_summary.to_csv(ONE_WAY_SUMMARY_CSV, index=False)
display(eligible_one_way[[
    "model_label", "temperature", "method", "attr_label", "party",
    "attribute_label", "category", "n_personas_total",
    "mean_signed_error", "se_signed_error",
]].head(24).round(3))


## Main-text Fig. 5 and SI Fig. S5

Categories are selected by the average magnitude of their configuration-level Mean
Signed Deviation (MSD). Negative values indicate underestimation and positive values
indicate overestimation. The same selected categories and x-axis limits are used at
both temperatures; method markers average over the two attribute orders and ten runs.


In [ ]:
# Add vote choice only for this plot, matching the scope of the earlier figure.
vote_column = "w2presvtwho"
vote_demo = exposure_demo.merge(
    survey[["caseid", vote_column]], on="caseid", how="left", validate="many_to_one"
)
vote_part = vote_demo[base_columns + [vote_column]].rename(
    columns={vote_column: "category"}
)
vote_part["attribute"] = vote_column
vote_part["attribute_label"] = "Vote choice"
vote_part["category"] = vote_part["category"].astype(str).str.strip()
vote_part = vote_part[vote_part["category"].ne("") & vote_part["category"].ne("nan")]
_, vote_summary = summarize_at_run_level(vote_part, one_way_groups)

signed_config_summary = pd.concat([one_way_summary, vote_summary], ignore_index=True)
signed_eligible = signed_config_summary[
    signed_config_summary["mean_valid_personas"].ge(MIN_PERSONAS)
].copy()

# Rank by the magnitude of directional bias across balanced configuration cells.
# The plotted x-coordinate remains signed.
signed_ranking = (
    signed_eligible.groupby(
        ["attribute", "attribute_label", "category", "party"],
        observed=True,
    )
    .agg(
        n_configuration_cells=("mean_signed_error", "count"),
        n_personas=("n_personas_total", "max"),
        mean_signed_error=("mean_signed_error", "mean"),
        mean_abs_configuration_msd=("mean_signed_error", lambda values: values.abs().mean()),
    ).reset_index()
)

N_CATEGORIES_PER_TARGET = 10
selected_rankings = (
    signed_ranking.sort_values(
        ["party", "mean_abs_configuration_msd"], ascending=[True, False]
    )
    .groupby("party", group_keys=False).head(N_CATEGORIES_PER_TARGET)
    .reset_index(drop=True)
)
selected_rankings.to_csv(SIGNED_RANKING_CSV, index=False)

METHOD_MARKERS = {"c1": "o", "c5": "s", "c10": "D", "c25": "^", "c50": "v"}
PARTY_COLORS = {"dem": "#2166ac", "rep": "#b2182b"}
MODEL_ORDER = [
    "Llama-3.1 8B", "Mistral 7B", "Qwen2.5 7B",
    "Llama-3.1 70B AWQ", "Qwen2.5 72B AWQ",
]


def compact_cell_label(attribute_label, category):
    replacements = {
        "bachelor's degree": "Bachelor's",
        "high school graduate": "High school",
        "less than high school": "Less than HS",
        "vocational tech school": "Vocational",
        "believer in a minority religion": "Minority religion",
        "Roman Catholic": "Catholic",
        "not religious": "Not religious",
        "didn't vote": "Did not vote",
    }
    category_label = replacements.get(category, category)
    prefixes = {
        "Age group": "Age ",
        "Race/ethnicity": "",
        "Education": "",
        "Religion": "",
        "Region": "",
        "Gender": "",
        "Vote choice": "Vote: ",
    }
    return prefixes.get(attribute_label, f"{attribute_label}: ") + category_label


selected_rankings["cell_label"] = selected_rankings.apply(
    lambda row: compact_cell_label(row["attribute_label"], row["category"]), axis=1
)

method_points = (
    signed_eligible.groupby(
        [
            "network_view", "temperature", "model_label", "method",
            "attribute", "attribute_label", "category", "party",
        ],
        observed=True,
    )
    .agg(mean_signed_error=("mean_signed_error", "mean"))
    .reset_index()
    .merge(
        selected_rankings[[
            "attribute", "attribute_label", "category", "party", "cell_label",
        ]],
        on=["attribute", "attribute_label", "category", "party"],
        how="inner", validate="many_to_one",
    )
)
model_averages = (
    method_points.groupby(
        [
            "network_view", "temperature", "model_label", "attribute",
            "attribute_label", "category", "party", "cell_label",
        ],
        observed=True,
    )["mean_signed_error"].mean().reset_index()
)

x_padding = .15
x_limit = max(
    abs(method_points["mean_signed_error"].min()),
    abs(method_points["mean_signed_error"].max()),
) + x_padding
x_limits = (-x_limit, x_limit)


def plot_largest_signed_errors(network_view, temperature):
    fig = plt.figure(figsize=(20, 14), constrained_layout=False)
    outer = fig.add_gridspec(2, 1, hspace=.27)

    for party_index, party in enumerate(["dem", "rep"]):
        party_cells = (
            selected_rankings[selected_rankings["party"].eq(party)]
            .sort_values("mean_abs_configuration_msd", ascending=False)
        )
        cell_order = party_cells["cell_label"].tolist()
        y_positions = {label: index for index, label in enumerate(cell_order)}
        grid = outer[party_index].subgridspec(1, len(MODEL_ORDER), wspace=.12)

        for model_index, model_label in enumerate(MODEL_ORDER):
            axis = fig.add_subplot(grid[0, model_index])
            subset = method_points[
                method_points["network_view"].eq(network_view)
                & np.isclose(method_points["temperature"], temperature)
                & method_points["model_label"].eq(model_label)
                & method_points["party"].eq(party)
            ]
            averages = model_averages[
                model_averages["network_view"].eq(network_view)
                & np.isclose(model_averages["temperature"], temperature)
                & model_averages["model_label"].eq(model_label)
                & model_averages["party"].eq(party)
            ]

            for separator in np.arange(.5, len(cell_order) - .5, 1):
                axis.axhline(
                    separator, color="0.82", linestyle=(0, (3, 3)),
                    linewidth=.7, zorder=0,
                )
            for method in METHOD_ORDER:
                points = subset[subset["method"].astype(str).eq(method)]
                axis.scatter(
                    points["mean_signed_error"],
                    points["cell_label"].map(y_positions),
                    s=42, marker=METHOD_MARKERS[method], color=PARTY_COLORS[party],
                    alpha=.38, edgecolor="none", zorder=2,
                )
            axis.scatter(
                averages["mean_signed_error"],
                averages["cell_label"].map(y_positions),
                s=92, marker="o", color=PARTY_COLORS[party], alpha=.96,
                edgecolor="black", linewidth=1, zorder=4,
            )
            axis.axvline(0, color="0.15", linewidth=1.1, zorder=1)
            axis.grid(axis="x", color="0.88", linewidth=.8)
            axis.set_xlim(x_limits)
            axis.set_ylim(len(cell_order) - .55, -.55)
            axis.set_title(
                model_label if party_index == 0 else "", fontsize=20, pad=10
            )
            axis.set_yticks(range(len(cell_order)))
            axis.set_yticklabels(cell_order if model_index == 0 else [])
            axis.set_xlabel("")
            axis.set_ylabel("")
            axis.tick_params(axis="x", labelsize=16)
            axis.tick_params(axis="y", labelsize=15, pad=6)
            for spine in ["top", "right", "left"]:
                axis.spines[spine].set_visible(False)

    fig.text(.065, .915, "a", fontsize=32, ha="left", va="bottom")
    fig.text(.10, .915, "Democratic exposure", fontsize=25, ha="left", va="bottom")
    fig.text(.065, .485, "b", fontsize=32, ha="left", va="bottom")
    fig.text(.10, .485, "Republican exposure", fontsize=25, ha="left", va="bottom")
    fig.text(
        .985, .94, f"{VIEW_LABELS[network_view]} · T={temperature:.1f}",
        fontsize=24, ha="right", va="top",
    )
    fig.text(
        .5, .055,
        "Mean Signed Deviation (MSD = LLM exposure category − survey exposure category)",
        ha="center", fontsize=21,
    )
    handles = [
        Line2D(
            [0], [0], marker="o", color="none", markerfacecolor=PARTY_COLORS[party],
            markeredgecolor="black", markersize=9,
            label=f"Model average: {PARTY_LABELS[party].lower()}",
        )
        for party in ["dem", "rep"]
    ] + [
        Line2D(
            [0], [0], marker=METHOD_MARKERS[method], color="0.45",
            markerfacecolor="0.45", linestyle="none", alpha=.65,
            markersize=8, label=method,
        )
        for method in METHOD_ORDER
    ]
    fig.legend(
        handles=handles, loc="lower center", ncol=7, frameon=False,
        bbox_to_anchor=(.5, .012), fontsize=15,
    )
    fig.subplots_adjust(left=.11, right=.985, top=.885, bottom=.115)
    if network_view == PRIMARY_NETWORK_VIEW and np.isclose(temperature, .1):
        paper_pdf = IMAGE_DIR / "Fig_5_group_specific_exposure_errors.pdf"
        fig.savefig(paper_pdf, bbox_inches="tight")
        print(f"Saved {paper_pdf.relative_to(BASE_DIR)}")
    if network_view == PRIMARY_NETWORK_VIEW and np.isclose(temperature, .7):
        si_stem = IMAGE_DIR / "Fig_S5_group_specific_exposure_errors_T07"
        fig.savefig(si_stem.with_suffix(".pdf"), bbox_inches="tight")
        fig.savefig(si_stem.with_suffix(".png"), dpi=300, bbox_inches="tight")
        print(f"Saved {si_stem.with_suffix('.pdf').relative_to(BASE_DIR)}")
    plt.show()


for view in NETWORK_VIEWS:
    for selected_temperature in sorted(method_points["temperature"].unique()):
        plot_largest_signed_errors(view, selected_temperature)

display(
    selected_rankings[[
        "party", "attribute_label", "category", "n_personas",
        "mean_signed_error", "mean_abs_configuration_msd",
    ]].round(3)
)
